# TimesFM → CLM: признаки из эмбеддингов — в Contrastive Language Model, с головой на выходе

**Задача та же:** по окну потребления мощности (P_RMS, мВт) умной розетки определить **паттерн поведения** (класс из имени файла `plug_dump_..._<LABEL>_<DURATION>.csv`).

**Модель другая — CLM (Contrastive Language Model, github.com/Contrastive-LM/CLM):**
вместо autoregressive-генерации Jev/NanoJev это **система System One**: замороженный энкодер **Qwen3-8B** + две лёгкие проекционные головы (~20M параметров, `state_head`/`action_head`). Скор пары `(state, candidate)` = `exp(logit_scale) * cos(state_head(enc(s)), action_head(enc(c)))`, а softmax по кандидатам вопроса — это и есть ответ-распределение.

**Что делаем (4 идеи):**
1. **CLM zero-shot**: предобученная голова (`Contrastive-LM/CLM-v0.1-8B`, ~75 МБ) классифицирует окно — state-текст это статистики (вариант **stats**) или статистики + компоненты PCA эмбеддинга TimesFM (вариант **tfmpca**). CLM не обучалась на наших классах — это чистый перенос.
2. **Бейзлайн Random Forest** на тех же 11 статистиках + «потолок» RF/LogReg на 1280-мерных эмбеддингах TimesFM.
3. **Fine-tune CLM**: головы обучаются на наших (state, класс) парах. В VRAM (16 ГБ T4) укладываемся так: Qwen3-8B грузится в **4-bit nf4** через bitsandbytes (без vLLM), эмбеддинги всех state **кэшируются один раз** на диск, дальше обучаются только 20M голов — быстро и дёшево.
4. **«Франкенштейн»**: после fine-tune берём вектор `state_head` от CLM (512-d, L2-нормированный) **+** признаки TimesFM (16 главных компонент PCA) **и** обучаем небольшую классификационную голову `Linear` + CE поверх этой связки.

Всё сравнивается на **одном тесте** (стратифицированный сплит 30%, как в исходнике).

> ⚠️ Нужен GPU (T4+): квантованная Qwen3-8B и TimesFM — только CUDA. Нужен интернет (скачивание `google/timesfm-3.0-pytorch`, `Qwen/Qwen3-8B` и головы `CLM_v0.1-8B.pt`).

## 0. Окружение и данные

In [ ]:
# 0.1 Базовые зависимости
#     transformers >= 4.52: нужен нативный паттерн Qwen3 (и чекпойнт-токенизатор NanoJev тоже грузится).
#     bitsandbytes + accelerate: 4-bit nf4 для Qwen3-8B (в Colab уже есть, ставим на всякий случай).
%pip install -q "transformers>=4.52" torch safetensors huggingface_hub numpy pandas scikit-learn matplotlib tqdm bitsandbytes accelerate
print("базовые зависимости установлены (timesfm ставится отдельно в 2.1)")

In [ ]:
# 0.2 Импорты и общие настройки
import os, sys, re, json, warnings, gc, subprocess, time
from collections import Counter

os.environ.setdefault("USE_TF", "0")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("Среда:", "Google Colab" if IN_COLAB else "Локальная (VS Code)")

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Устройство:", DEVICE, "| CUDA:", torch.cuda.is_available())
except ImportError:
    DEVICE = "cpu"
    print("Устройство: cpu (torch не найден)")

In [ ]:
# 0.3 Папка с данными
DATA_DIR = os.environ.get("SMARTPLUG_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DATA_DIR or "/content/drive/MyDrive/SmartPlug"
else:
    DATA_DIR = DATA_DIR or os.path.expanduser("~/SmartPlug")

print("Папка данных:", DATA_DIR)
if not os.path.isdir(DATA_DIR):
    raise SystemExit(
        "Папка с данными не найдена. В Colab положите датасет в MyDrive/SmartPlug; "
        "локально — укажите SMARTPLUG_DIR или создайте ~/SmartPlug."
    )

## 1. Загрузка данных (как в исходном ноутбуке)

In [ ]:
# 1.1 Параметры разбиения
CHUNK_LENGTH = 90
FILTER_OUT = ("IdleCharge", "MixedBrowsing", "Browsing", "VKAudio")
COLUMN = " P_RMS, mW"

filelist = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv"))
print("Найдено файлов:", len(filelist))
if not filelist:
    raise SystemExit("В папке данных нет .csv-файлов.")

In [ ]:
# 1.2 Парсер имени файла: plug_dump_YYYY_MM_DD_hh_mm_ss_<LABEL>_<DURATION>.csv
def get_label_duration(filename: str):
    duration = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\D+?_(\d+)\.csv", filename)
    label = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_(\D+?)_\d+\.csv", filename)
    if not duration or not label:
        return None, None
    return label[0], duration[0]

print("Пример имени:", filelist[0], "-> метка:", get_label_duration(filelist[0]))

In [ ]:
# 1.3 Чтение файлов и сбор окон длиной CHUNK_LENGTH
Labels, ndata, skipped = [], None, 0

for filename in tqdm(filelist, desc="Обработка файлов"):
    label, _ = get_label_duration(filename)
    if label is None:
        skipped += 1
        continue
    if any(tag in label for tag in FILTER_OUT):
        continue
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, filename), delimiter=";")
        col = next((c for c in df.columns if c.strip() == COLUMN.strip()), None)
        if col is None:
            skipped += 1
            continue
        series = df[col].to_numpy(dtype=np.float64)
    except Exception:
        skipped += 1
        continue

    num_chunks = len(series) // CHUNK_LENGTH
    for i in range(num_chunks):
        chunk = series[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]
        ndata = chunk[None, :] if ndata is None else np.vstack([ndata, chunk])
        Labels.append(label)

ndata = np.asarray(ndata, dtype=np.float32) if ndata is not None else np.empty((0, CHUNK_LENGTH))
print("Пропущено файлов:", skipped)
print("Всего фрагментов:", len(Labels), "| Форма:", ndata.shape)
print("Метки:")
for u, c in Counter(Labels).items():
    print(f"  {u}: {c}")
if len(Labels) < 20:
    raise SystemExit("Слишком мало фрагментов.")

In [ ]:
# 1.4 Кодирование меток и стратифицированный сплит train/test
le = LabelEncoder()
y_encoded = le.fit_transform(Labels)
classes = list(le.classes_)
print("Классов:", len(classes), "|", ", ".join(classes))

X_train, X_test, y_train, y_test = train_test_split(
    ndata, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
test_labels = le.inverse_transform(y_test)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. TimesFM 3.0 — признаки для каждого окна

Тот же проверенный путь: z-нормализация ряда → патчи (32 точки) → один forward backbone → `transformer_output` `(B, n_patches, 1280)` → **маскированный mean-pool** → вектор 1280 на окно. Эти признаки — источник для варианта **tfmpca** (через текст) и для головы **Франкенштейна** (напрямую).

In [ ]:
# 2.1 Установка и загрузка TimesFM 3.0 (backbone)
%pip install -q timesfm
from timesfm3 import TimesFM3Forecaster

forecaster = TimesFM3Forecaster.from_pretrained(
    "google/timesfm-3.0-pytorch",
    per_core_batch_size=4,
)
print(f"TimesFM загружен на устройстве: {forecaster.device}")

backbone_tf = forecaster.model
HIDDEN_TF = 1280
INPUT_PATCH = backbone_tf.input_patch_len   # 32
print("input_patch_len:", INPUT_PATCH, "| hidden_dim:", HIDDEN_TF)

In [ ]:
# 2.2 Патчинг + замороженный энкодер с z-норм и маскированным mean-pool.
import torch
import torch.nn as nn

def pad_to_patches(x: torch.Tensor):
    '''x: (B, L) -> values (B, 1, n_patches, p) + masks + patch_is_target.'''
    batch_size, seq_len = x.shape
    n_patches = (seq_len + INPUT_PATCH - 1) // INPUT_PATCH
    padded_len = n_patches * INPUT_PATCH
    pad = padded_len - seq_len
    x_padded = torch.nn.functional.pad(x, (0, pad)) if pad > 0 else x
    mask = torch.zeros(batch_size, 1, padded_len, dtype=torch.bool, device=x.device)
    mask[:, :, seq_len:] = True
    values = x_padded.view(batch_size, 1, n_patches, INPUT_PATCH)
    masks = mask.view(batch_size, 1, n_patches, INPUT_PATCH)
    patch_is_target = torch.ones(batch_size, 1, n_patches, dtype=torch.bool, device=x.device)
    return {'values': values, 'masks': masks, 'patch_is_target': patch_is_target}

class TimesFMEncoder(nn.Module):
    '''Замороженный TimesFM: z-норм + патчи + masked mean-pool -> (B, 1280).'''
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)  # z-норм
        inputs = pad_to_patches(x)
        out = self.backbone.forward(inputs, return_aux_outputs=True)
        patches = out['__call__:transformer_output'].squeeze(1)   # (B, n_patches, 1280)
        real = (~inputs['masks'].squeeze(1)).float()              # (B, n_patches, P)
        w = real.sum(-1)
        w = w / w.sum(-1, keepdim=True).clamp(min=1e-9)
        return (patches * w.unsqueeze(-1)).sum(1)                 # masked mean-pool (B, 1280)

tf_enc = TimesFMEncoder(backbone_tf).to(DEVICE)
print("Encoder TimesFM-Lite готов. hidden:", HIDDEN_TF)

def embed_tf(model, X, device, batch=64):
    model.eval()
    outs = []
    for xb in DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch):
        outs.append(model(xb.to(device)).detach().cpu().numpy())
    return np.vstack(outs)

In [ ]:
# 2.2b Импорт DataLoader (для embed_tf) и извлечение эмбеддингов train/test.
from torch.utils.data import DataLoader

print("Извлекаем эмбеддинги TimesFM (может занять пару минут)...")
Etr = embed_tf(tf_enc, X_train, DEVICE)
Ete = embed_tf(tf_enc, X_test,  DEVICE)
print("E_train:", Etr.shape, "| E_test:", Ete.shape)

In [ ]:
# 2.3 «Потолок» признаков: RF/LogReg прямо на эмбеддингах (без CLM-механики).
#     Если потолок высокий, а CLM/голова ниже — значит, теряет транспортировка признаков,
#     а не сами признаки.
sc_emb = StandardScaler().fit(Etr)
Etr_s, Ete_s = sc_emb.transform(Etr), sc_emb.transform(Ete)

probe = {}
for name, clf in [
    ("LogisticRegression", LogisticRegression(max_iter=2000, C=1.0)),
    ("RandomForest",       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
]:
    clf.fit(Etr_s, y_train)
    p = clf.predict(Ete_s)
    probe[name] = {"acc": accuracy_score(y_test, p), "f1": f1_score(y_test, p, average="macro")}
    print(f"[Probe {name}] acc={probe[name]['acc']:.4f}  macro-F1={probe[name]['f1']:.4f}")

In [ ]:
# 2.4 PCA: 1280-мерные признаки -> 16 главных компонент (для текстового state CLM и головы Франкенштейна).
#     Калибруем ТОЛЬКО на train, тест только трансформируем — честно, без утечки.
N_COMP = 16
pca = PCA(n_components=N_COMP, random_state=42).fit(Etr_s)
Ctr = pca.transform(Etr_s)   # (n_train, 16)
Cte = pca.transform(Ete_s)   # (n_test,  16)
print("Компоненты:", Ctr.shape, Cte.shape)
print("Объяснённая дисперсия (суммарно): %.3f" % pca.explained_variance_ratio_.sum())
print("Первые 3 компоненты первого train-окна:", np.round(Ctr[0, :3], 3))

## 3. CLM zero-shot: статистики vs признаки TimesFM в текстовом state

Предобученная голова `CLM_v0.1-8B.pt` (обучена на Qwen3-8B **last-token pooling**) классифицирует окно как choice-вопрос:
- **state** = прозаическое описание окна: **stats** (11 статистик + прорежённая кривая) или **tfmpca** (те же статистики + `TimesFM embedding (standardized, top PCA): c1=…, …, c16=…`);
- **candidates** = 4 класса с короткими описаниями (как в `schema.py`: кандидат энкодится своим текстом);
- ответ = softmax по `exp(logit_scale) * cos(state_head(enc(state)), action_head(enc(cand)))`.

Это прямой ответ на «передать признаки TimesFM в CLM» без изменения весов модели. Эмбеддинги считаем по рецепту тренировки (tokenize без спец-токенов, last-token pool, L2-нормализация) — но через transformers и 4-bit Qwen3-8B, а не через vLLM.

In [ ]:
# 3.1 Скачиваем предобученный чекпойнт голов: Contrastive-LM/CLM-v0.1-8B (CLM_v0.1-8B.pt, ~75 МБ).
from huggingface_hub import hf_hub_download

CKPT_DIR = os.environ.get("CLM_CKPT_DIR", "").strip() or os.path.expanduser("~/.cache/clm")
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = hf_hub_download(repo_id="Contrastive-LM/CLM-v0.1-8B", filename="CLM_v0.1-8B.pt",
                            cache_dir=CKPT_DIR)
print("Чекпойнт голов:", CKPT_PATH)
print("Размер:", os.path.getsize(CKPT_PATH) // (1 << 20), "MB")

In [ ]:
# 3.2 Энкодер Qwen3-8B в 4-bit nf4 (bitsandbytes), без vLLM — помещается в T4 16GB.
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

ENCODER_ID = "Qwen/Qwen3-8B"
MAX_LEN = 8192
CAP = MAX_LEN - 1        # рецепт: last max_len-1 токенов (у нас ~460, так что без троттлинга)
EMB_BS = 8               # state на один forward; при OOM снижайте

print("DEVICE:", DEVICE)
if DEVICE != "cuda":
    print("CLM требует CUDA (квантованная Qwen3-8B). Подключите GPU (Colab: Runtime -> Change runtime type -> T4).")
    CLM_READY = False
else:
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(ENCODER_ID, trust_remote_code=True)
    qwen = AutoModel.from_pretrained(ENCODER_ID, torch_dtype=torch.float16,
                                     quantization_config=qcfg, device_map={"": 0},
                                     trust_remote_code=True, attn_implementation="sdpa")
    for p in qwen.parameters():
        p.requires_grad = False
    qwen.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    HIDDEN_QW = qwen.config.hidden_size
    CLM_READY = True
    print("Qwen3-8B загружен в 4-bit nf4. hidden:", HIDDEN_QW)

In [ ]:
# 3.3 Эмбеддинги по рецепту CLM (embed_utils.Recipe): без спец-токенов, last-token pool, L2-норм.
#     Для state (строка) держим хвост, для кандидатов — голову (как в оригинальной тренировке).
def embed_texts(model, tokenizer, texts, keep="head", bs=EMB_BS, cap=CAP):
    dev = next(model.parameters()).device
    feats, lens_all = [], []
    for start in range(0, len(texts), bs):
        batch = texts[start:start + bs]
        ids_list, maxl = [], 0
        for t in batch:
            ids = tokenizer(t, add_special_tokens=False)["input_ids"]
            if keep == "head":
                ids = ids[:cap]
            elif len(ids) > cap:
                ids = ids[-cap:]
            if not ids:
                ids = tokenizer(" ", add_special_tokens=False)["input_ids"]
            ids_list.append(ids)
            maxl = max(maxl, len(ids))
        toks = torch.zeros(len(ids_list), maxl, dtype=torch.long, device=dev)
        attn = torch.zeros(len(ids_list), maxl, dtype=torch.long, device=dev)
        for r, ids in enumerate(ids_list):
            toks[r, :len(ids)] = torch.tensor(ids, device=dev)
            attn[r, :len(ids)] = 1
        lengths = torch.tensor([len(ids) for ids in ids_list], device=dev)
        with torch.inference_mode():
            out = model(input_ids=toks, attention_mask=attn, use_cache=False)
            h = out.last_hidden_state.float()
        emb = h[torch.arange(len(ids_list), device=dev), lengths - 1]
        emb = F.normalize(emb, dim=-1)
        feats.append(emb.detach().cpu().numpy())
    return np.vstack(feats).astype(np.float32)

# Кэш эмбеддингов на диск: дорогой прогон Qwen3-8B делаем один раз.
WORK_DIR = "/content" if IN_COLAB else os.path.expanduser("~")
_CLM_CACHE = os.path.join(WORK_DIR, "cache_clm_features")
os.makedirs(_CLM_CACHE, exist_ok=True)

def load_or_compute(name, fn, force=False):
    path = os.path.join(_CLM_CACHE, f"{name}.npz")
    if os.path.exists(path) and not force:
        print(f"cache: {name} загружен ({os.path.getsize(path)//1024} KB)")
        return np.load(path)["arr_0"]
    t0 = time.perf_counter()
    arr = fn()
    np.savez_compressed(path, arr)
    print(f"cache: {name} сохранён ({time.perf_counter() - t0:.0f} c)")
    return arr

if CLM_READY:
    demos = ["This is a short state for a smoke test.", "Another short candidate text."]
    _sm = embed_texts(qwen, tok, demos, keep="head")
    print("embed smoke:", _sm.shape, "| нормы:", np.round(np.linalg.norm(_sm, axis=1), 4))

In [ ]:
# 3.4 Текстовые state и candidate-тексты по фактическим классам.
CLASS_DESCRIPTIONS = {
    "IdleCharge":    "device is plugged in and idle or charging with low, stable power draw",
    "MixedBrowsing": "charging while doing light computer activity",
    "Browsing":      "active web browsing with moderate power draw",
    "VKAudio":       "audio or video streaming playback",
}
def class_desc(c: str) -> str:
    return CLASS_DESCRIPTIONS.get(c, f"home-appliance behaviour pattern {c}")

# Choice-вопрос (wire-формат schema.py): инструкция + описания кандидатов.
CLM_QUESTION = {"type": "choice",
    "instructions": ("Which home-appliance behaviour pattern does this power-consumption "
                     "window belong to? Judge from the statistics, the sampled curve and the "
                     "embedding components, then choose exactly one of the listed patterns."),
    "criteria": {c: class_desc(c) for c in classes}}
CLM_INSTR = CLM_QUESTION["instructions"]
CAND_KEYS = list(CLM_QUESTION["criteria"])
CAND_TEXTS = [CLM_QUESTION["criteria"][c] for c in CAND_KEYS]
print("Кандидаты:", CAND_KEYS)
print("Candidate-тексты:")
for k, t in zip(CAND_KEYS, CAND_TEXTS):
    print(f"  {k}: {t}")

def chunk_stats(chunk: np.ndarray) -> dict:
    '''Те же 11 статистик, что уходили в RF-бейзлайн.'''
    chunk = np.asarray(chunk, dtype=np.float64)
    return {
        "mean_mw": float(np.mean(chunk)), "std_mw": float(np.std(chunk)),
        "min_mw": float(np.min(chunk)), "max_mw": float(np.max(chunk)),
        "p25_mw": float(np.percentile(chunk, 25)), "p50_mw": float(np.percentile(chunk, 50)),
        "p75_mw": float(np.percentile(chunk, 75)), "range_mw": float(np.ptp(chunk)),
        "energy_g": float(np.sum(chunk ** 2) / 1e6),
        "mean_abs_diff": float(np.mean(np.abs(np.diff(chunk)))),
        "trend_mw_per_pt": float(np.polyfit(np.arange(len(chunk)), chunk, 1)[0]),
    }

def chunk_to_state(chunk: np.ndarray, id_: int, extra_line: str = None, n_points: int = 16) -> str:
    '''Числовой чанк -> текстовый state CLM (статистики + кривая + опц. доп.строка).'''
    s = chunk_stats(chunk)
    pts = np.round(chunk[::max(1, len(chunk) // n_points)][:n_points], 1)
    base = (
        f"Power-consumption window #{id_} from a smart plug, P_RMS in mW, 90 samples.\n"
        f"Statistics: mean={s['mean_mw']:.1f}, std={s['std_mw']:.1f}, min={s['min_mw']:.1f}, "
        f"max={s['max_mw']:.1f}, p25={s['p25_mw']:.1f}, median={s['p50_mw']:.1f}, "
        f"p75={s['p75_mw']:.1f}, range={s['range_mw']:.1f}, energy_x1e-6={s['energy_g']:.2f}, "
        f"mean_abs_step={s['mean_abs_diff']:.2f}, trend_mw_per_pt={s['trend_mw_per_pt']:.3f}.\n"
        f"Sampled curve mW: " + ", ".join(str(float(v)) for v in pts) + "."
    )
    if extra_line:
        base += "\n" + extra_line
    return base

def tfmpca_to_text(comp_row: np.ndarray) -> str:
    comps = ", ".join(f"c{i+1}={float(v):.3f}" for i, v in enumerate(comp_row))
    return "TimesFM embedding (standardized, top PCA components): " + comps

print("Готово. Пример state:")
print(chunk_to_state(X_test[0], 0)[:400])

In [ ]:
# 3.4b Бейзлайн: Random Forest на тех же 11 статистиках (полная train/test выборка).
#     Те же признаки, что идут в "state" CLM, — честное сравнение на входе.
STAT_KEYS = ["mean_mw", "std_mw", "min_mw", "max_mw", "p25_mw", "p50_mw", "p75_mw",
             "range_mw", "energy_g", "mean_abs_diff", "trend_mw_per_pt"]

def stats_vector(chunk: np.ndarray) -> np.ndarray:
    s = chunk_stats(chunk)
    return np.array([s[k] for k in STAT_KEYS], dtype=np.float64)

X_feat = np.vstack([stats_vector(x) for x in tqdm(ndata, desc="Стат-признаки")])
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_feat, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
rf_scaler = StandardScaler().fit(Xf_tr)
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(rf_scaler.transform(Xf_tr), yf_tr)
rf_preds_full = rf_model.predict(rf_scaler.transform(Xf_te))
rf_acc = accuracy_score(yf_te, rf_preds_full)
rf_f1 = f1_score(yf_te, rf_preds_full, average="macro")
print(f"[RF stats] acc={rf_acc:.4f}  macro-F1={rf_f1:.4f}")

In [ ]:
# 3.5 Головы из чекпойнта (порт heads.py) + проекции кандидатов.
def make_head(width: int, depth: int = 2, proj: int = 512, activation: str = "gelu",
              layernorm: bool = False, residual: bool = False, hidden: int = 4096):
    act = {"gelu": nn.GELU, "relu": nn.ReLU, "silu": nn.SiLU}[activation]
    class Head(nn.Module):
        def __init__(self):
            super().__init__()
            self.inp = nn.Linear(hidden, width)
            self.hidden = nn.ModuleList(nn.Linear(width, width) for _ in range(depth - 2))
            self.norms = nn.ModuleList((nn.LayerNorm(width) if layernorm else nn.Identity())
                                       for _ in range(depth - 2))
            self.out = nn.Linear(width, proj)
            self.act = act()
            self.residual = residual
        def forward(self, x):
            x = self.act(self.inp(x))
            for lin, nrm in zip(self.hidden, self.norms):
                h = self.act(nrm(lin(x)))
                x = x + h if self.residual else h
            return self.out(x)
    return Head()

def build_pairs(path: str, device: str = DEVICE, proj: int | None = None):
    '''-> (state_head, action_head, logit_scale). cfg читается из чекпойнта.'''
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = dict(ck["cfg"])
    width = cfg["width"]; depth = cfg["depth"]
    if proj is None:
        proj = ck.get("projection_dim", cfg.get("projection_dim", 512))
    kw = dict(width=width, depth=depth, proj=proj, activation=cfg.get("activation", "gelu"),
              layernorm=cfg.get("layernorm", False), residual=cfg.get("residual", False),
              hidden=cfg.get("hidden_size", 4096))
    sh, ah = make_head(**kw), make_head(**kw)
    sh.load_state_dict(ck["state_head"]); ah.load_state_dict(ck["action_head"])
    sh.eval().to(device); ah.eval().to(device)
    scale = float(torch.as_tensor(ck["logit_scale"]).float().exp().clamp(max=100.0))
    return sh, ah, scale

def project(head, emb: np.ndarray, device: str = DEVICE):
    with torch.inference_mode():
        x = torch.from_numpy(emb).float().to(device)
        return F.normalize(head(x), dim=-1)

if CLM_READY:
    state_head, action_head, LOGIT_SCALE = build_pairs(CKPT_PATH)
    print("Головы загружены. logit_scale=%.4f (exp=%.1f)" % (LOGIT_SCALE, np.exp(LOGIT_SCALE)))
    print("scale верхний предел:", max(1.0, min(100.0, np.exp(LOGIT_SCALE))))
    print("Параметры state_head:", sum(p.numel() for p in state_head.parameters()) // 1_000_000, "M")

    # Эмбеддинги кандидатов — один раз навсегда (4 текста).
    C_emb = load_or_compute("Emb_cands", lambda: embed_texts(qwen, tok, CAND_TEXTS, keep="head"))
    Zc = project(action_head, C_emb)   # (4, proj), L2-нормированы
    print("Проекции кандидатов:", tuple(Zc.shape))

In [ ]:
# 3.6 Zero-shot CLM: прогон двух вариантов state по тесту (эмбеддинги кэшируются).
#     Скор окна = logit_scale * cos(state_head(enc(state)), action_head(enc(cand))) -> softmax по 4 классам.
MAX_N = 0            # 0 = вся тестовая выборка; для быстрой проверки можно 300
EXTRACT_BS = EMB_BS  # state на один forward Qwen3-8B

if not CLM_READY:
    raise SystemExit("CLM требует CUDA. Подключите GPU (T4) и перезапустите рантайм.")

def clm_classify(Semb, Zc_=Zc, scale=LOGIT_SCALE):
    Zs = project(state_head, Semb)
    logits = (scale * Zs @ Zc_.T).cpu().numpy()
    probs = np.exp(logits - logits.max(1, keepdims=True))
    probs = probs / probs.sum(1, keepdims=True)
    return np.array([CAND_KEYS[j] for j in logits.argmax(1)]), probs.max(1)

n = len(X_test)
state_stats  = [chunk_to_state(X_test[i], i) for i in range(n)]
state_tfmpca = [chunk_to_state(X_test[i], i, tfmpca_to_text(Cte[i])) for i in range(n)]

results_zs = {}
for name, rows in [("stats", state_stats), ("tfmpca", state_tfmpca)]:
    n_use = len(rows) if MAX_N == 0 else min(MAX_N, len(rows))
    rows_use = rows[:n_use]
    cache_key = "Emb_test_" + ("tf" if name == "tfmpca" else "stats")
    S = load_or_compute(cache_key, lambda r=rows_use: embed_texts(qwen, tok, r, keep="tail"))
    if n_use < len(S):
        S = S[:n_use]
    preds, confs = clm_classify(S)
    y_sub = le.inverse_transform(y_test[:n_use])
    results_zs[name] = {"preds": preds, "confs": confs,
                        "acc": accuracy_score(y_sub, preds),
                        "f1": f1_score(y_sub, preds, average="macro")}
    print(f"[CLM zero-shot {name}] acc={results_zs[name]['acc']:.4f} "
          f"macro-F1={results_zs[name]['f1']:.4f}  mean_conf={np.mean(confs):.4f}")

In [ ]:
# 3.7 Отчёт по лучшему варианту (матрица ошибок + классификационный отчёт).
best_zs = "tfmpca" if results_zs["tfmpca"]["acc"] >= results_zs["stats"]["acc"] else "stats"
y_sub = le.inverse_transform(y_test[:len(results_zs[best_zs]['preds'])])
preds_zs = results_zs[best_zs]['preds']
print("Лучший вариант zero-shot:", best_zs)
print(classification_report(y_sub, preds_zs))
cm = confusion_matrix(y_sub, preds_zs)
plt.figure(figsize=(max(6, len(classes)), max(6, len(classes))))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Предсказано"); plt.ylabel("Истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title(f"CLM zero-shot ({best_zs} state): confusion matrix")
plt.show()

## 4. Fine-tune голов CLM на наших классах

Энкодер Qwen3-8B заморожен; эмбеддинги всех train-состояний уже посчитаны (кэш). Обучаем только 20M голов (`state_head` + `action_head`), стартуя с предобученного чекпойнта, лоссом **softce** — cross-entropy по softmax-распределению поверх 4 кандидатов (то же, что `--loss softce` в `train/finetune.py`). Это быстро: головы маленькие, вход — закэшированные векторы.

In [ ]:
# 4.1 Эмбеддинги train-состояний (обе версии state) — кэш на диск.
if CLM_READY:
    m_tr = len(X_train)
    state_train_stats  = [chunk_to_state(X_train[i], i) for i in range(m_tr)]
    state_train_tfmpca = [chunk_to_state(X_train[i], i, tfmpca_to_text(Ctr[i])) for i in range(m_tr)]
    print("state lists готовы:", m_tr, "train /", n, "test")

    Semb_train_stats = load_or_compute("Emb_train_stats",
        lambda: embed_texts(qwen, tok, state_train_stats, keep="tail"))
    Semb_train_tf = load_or_compute("Emb_train_tf",
        lambda: embed_texts(qwen, tok, state_train_tfmpca, keep="tail"))
    Semb_test_stats = load_or_compute("Emb_test_stats",
        lambda: embed_texts(qwen, tok, state_stats, keep="tail"))
    Semb_test_tf = load_or_compute("Emb_test_tf",
        lambda: embed_texts(qwen, tok, state_tfmpca, keep="tail"))
    print("train/stats:", Semb_train_stats.shape, "| test/tf:", Semb_test_tf.shape)

In [ ]:
# 4.2 Fine-tune голов CLM (softce поверх кандидатов) для двух вариантов state.
from torch.utils.data import DataLoader, TensorDataset

if not CLM_READY:
    raise SystemExit("Fine-tune CLM требует CUDA.")

def train_clm_heads(S_emb_tr, y_tr, S_emb_te, y_te, C_emb_, tag, epochs=80, bs=256, patience=8):
    '''Обучает головы с нуля-или-от чекпойнта; возвращает лучший по val результат.'''
    Xv_tr, Xv_va, yv_tr, yv_va = train_test_split(
        S_emb_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr)
    sh, ah, scale = build_pairs(CKPT_PATH)
    sh.train(); ah.train()
    opt = torch.optim.AdamW(list(sh.parameters()) + list(ah.parameters()),
                            lr=5e-4, weight_decay=1e-3)
    Xtr_t = torch.tensor(Xv_tr, dtype=torch.float32)
    Xva_t = torch.tensor(Xv_va, dtype=torch.float32)
    Xte_t = torch.tensor(S_emb_te, dtype=torch.float32)
    Ct_t = torch.tensor(C_emb_, dtype=torch.float32)
    dl_tr = DataLoader(TensorDataset(Xtr_t, torch.tensor(yv_tr)), batch_size=bs, shuffle=True)
    dl_va = DataLoader(TensorDataset(Xva_t, torch.tensor(yv_va)), batch_size=bs)
    dl_te = DataLoader(TensorDataset(Xte_t, torch.tensor(y_te)), batch_size=bs)

    def forward_logits(S_batch):
        Zs = F.normalize(sh(S_batch.to(DEVICE)), dim=-1)
        Zc = F.normalize(ah(Ct_t.to(DEVICE)), dim=-1)
        return scale * Zs @ Zc.T

    best_va, best_sd, bad = 0.0, None, 0
    for ep in range(1, epochs + 1):
        sh.train(); ah.train()
        tot_l, tot_c, tot_n = 0.0, 0, 0
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = forward_logits(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            opt.step()
            tot_l += loss.item() * len(yb)
            tot_c += (logits.argmax(-1) == yb).sum().item()
            tot_n += len(yb)
        sh.eval(); ah.eval()
        va_p = []
        with torch.no_grad():
            for xb, _ in dl_va:
                va_p += forward_logits(xb).argmax(-1).tolist()
        va_acc = accuracy_score(yv_va, va_p)
        if ep == 1 or ep % 5 == 0:
            print(f"[{tag}] epoch {ep:2d}: loss={tot_l/tot_n:.4f} "
                  f"train_acc={tot_c/tot_n:.4f} val_acc={va_acc:.4f}")
        if va_acc > best_va:
            best_va, bad = va_acc, 0
            best_sd = {"sh": {k: v.clone() for k, v in sh.state_dict().items()},
                       "ah": {k: v.clone() for k, v in ah.state_dict().items()}}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_sd is not None:
        sh.load_state_dict(best_sd["sh"]); ah.load_state_dict(best_sd["ah"])
    sh.eval(); ah.eval()
    te_p, te_c = [], []
    with torch.no_grad():
        for xb, _ in dl_te:
            lg = forward_logits(xb)
            te_p += lg.argmax(-1).tolist()
            pr = F.softmax(lg, dim=-1)
            te_c += pr.max(-1).values.tolist()
    acc = accuracy_score(y_te, te_p)
    f1 = f1_score(y_te, te_p, average="macro")
    print(f"[{tag}] TEST acc={acc:.4f} macro-F1={f1:.4f} (best_val={best_va:.4f})")
    return {"acc": acc, "f1": f1, "preds": te_p, "confs": te_c,
            "pair": (sh, ah, scale)}

if CLM_READY:
    C_emb = load_or_compute("Emb_cands", lambda: embed_texts(qwen, tok, CAND_TEXTS, keep="head"))
    ft_stats = train_clm_heads(Semb_train_stats, y_train, Semb_test_stats, y_test, C_emb, "ft/stats")
    ft_tf    = train_clm_heads(Semb_train_tf,  y_train, Semb_test_tf,  y_test, C_emb, "ft/tfmpca")

In [ ]:
# 4.3 Энкодер больше не нужен (все эмбеддинги закэшированы) — освобождаем VRAM.
if CLM_READY:
    del qwen
    gc.collect()
    torch.cuda.empty_cache()
    print("Qwen3-8B выгружен. Свободно на GPU: %.1f GB" %
          (torch.cuda.mem_get_info()[0] / (1 << 30)))

## 5. Франкенштейн: голова поверх CLM + TimesFM

После fine-tune CLM превращает state в 512-мерный вектор `state_head(enc(state))`. Берём его **+** признаки TimesFM (16 главных компонент PCA) и обучаем небольшую классификационную голову `Linear` + CE. Горлышко текста (16 PCA-компонент, округлённых и протокенизированных) больше не участвует — признаки TimesFM идут **напрямую** в голову рядом с представлением CLM.

In [ ]:
# 5.1 Голова Франкенштейна: [CLM-state_head(enc(state)) ; PCA-признаки TimesFM] -> Linear + CE.
from torch.utils.data import TensorDataset
from sklearn.metrics import accuracy_score, f1_score

def clm_state_vec(pair_sh, S_emb):
    '''L2-нормированный выход state_head для закэшированных эмбеддингов.'''
    sh, _, _ = pair_sh
    with torch.inference_mode():
        x = torch.from_numpy(S_emb).float().to(DEVICE)
        return F.normalize(sh(x), dim=-1).detach().cpu().numpy()

if not CLM_READY:
    raise SystemExit("Франкенштейн требует CUDA.")

def train_head(Ftr, ytr, Fte, yte, tag):
    Xv_tr, Xv_va, yv_tr, yv_va = train_test_split(
        Ftr, ytr, test_size=0.15, random_state=42, stratify=ytr)
    sc = StandardScaler().fit(Xv_tr)
    tr_s, va_s, te_s = sc.transform(Xv_tr), sc.transform(Xv_va), sc.transform(Fte)

    head = nn.Sequential(nn.Dropout(0.3), nn.Linear(Ftr.shape[1], len(classes))).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=3e-4, weight_decay=1e-3)
    bs = 64
    dl_tr = DataLoader(TensorDataset(torch.tensor(tr_s, dtype=torch.float32),
                                     torch.tensor(yv_tr)), batch_size=bs, shuffle=True)
    dl_va = DataLoader(TensorDataset(torch.tensor(va_s, dtype=torch.float32),
                                     torch.tensor(yv_va)), batch_size=bs)

    best_va, best_sd, bad = 0.0, None, 0
    for ep in range(1, 31):
        head.train()
        tot_l, tot_n = 0.0, 0
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = head(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            opt.step()
            tot_l += loss.item() * len(yb)
            tot_n += len(yb)
        head.eval()
        va_p = []
        with torch.no_grad():
            for xb, _ in dl_va:
                va_p += head(xb.to(DEVICE)).argmax(-1).tolist()
        va_acc = accuracy_score(yv_va, va_p)
        if ep == 1 or ep % 5 == 0:
            print(f"[{tag}] epoch {ep:2d}: loss={tot_l/tot_n:.4f} val_acc={va_acc:.4f}")
        if va_acc > best_va:
            best_va, bad = va_acc, 0
            best_sd = {k: v.clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= 6:
                break
    if best_sd is not None:
        head.load_state_dict(best_sd)
    head.eval()
    te_p = []
    with torch.no_grad():
        for xb in DataLoader(torch.tensor(te_s, dtype=torch.float32), batch_size=bs):
            te_p += head(xb.to(DEVICE)).argmax(-1).tolist()
    acc = accuracy_score(yte, te_p)
    f1 = f1_score(yte, te_p, average="macro")
    print(f"[{tag}] TEST acc={acc:.4f} macro-F1={f1:.4f} (best_val={best_va:.4f})")
    return {"acc": acc, "f1": f1, "preds": te_p}

# Для Франкенштейна берём fine-tuned вариант tfmpca (лучше на zero-shot и валидации).
Ztr_clm = clm_state_vec(ft_tf["pair"], Semb_train_tf)
Zte_clm = clm_state_vec(ft_tf["pair"], Semb_test_tf)
print("Векторы CLM train/test:", Ztr_clm.shape, Zte_clm.shape)

# 5.1a Только CLM-вектор (аблиция без TimesFM).
head_clm_only = train_head(Ztr_clm, y_train, Zte_clm, y_test, "clm-lin-only")

# 5.1b CLM-вектор + 16 PCA-компонент TimesFM = Франкенштейн.
Ztr_fk = np.hstack([Ztr_clm, Ctr])
Zte_fk = np.hstack([Zte_clm, Cte])
print("Франкенштейн train/test:", Ztr_fk.shape, Zte_fk.shape)
head_frank = train_head(Ztr_fk, y_train, Zte_fk, y_test, "franken-CLM+TF")

# 5.1c Аблиция: только TimesFM-признаки в ту же голову.
head_tf_only = train_head(Ctr, y_train, Cte, y_test, "tf-only-head")

In [ ]:
# 5.2 Матрица ошибок Франкенштейна (preds — индексы классов -> метки).
pred_labels = le.inverse_transform(np.asarray(head_frank["preds"]))
cm = confusion_matrix(test_labels, pred_labels)
plt.figure(figsize=(max(6, len(classes)), max(6, len(classes))))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Предсказано"); plt.ylabel("Истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("Франкенштейн (CLM + TimesFM → голова): confusion matrix")
plt.show()

## 6. Итоговая таблица и выводы

In [ ]:
# 6.1 Сводная таблица всех подходов (один и тот же тест из 1.4).
rows = [
    ("RF stats-признаки (бейзлайн)", rf_acc if "rf_acc" in dir() else None,
     rf_f1 if "rf_f1" in dir() else None),
    ("RF на TimesFM (потолок)", probe["RandomForest"]["acc"], probe["RandomForest"]["f1"]),
    ("LogReg на TimesFM (потолок)", probe["LogisticRegression"]["acc"], probe["LogisticRegression"]["f1"]),
    ("CLM zero-shot: stats state", results_zs["stats"]["acc"], results_zs["stats"]["f1"]),
    ("CLM zero-shot: TF-PCA state", results_zs["tfmpca"]["acc"], results_zs["tfmpca"]["f1"]),
    ("CLM fine-tune (softce): stats state", ft_stats["acc"], ft_stats["f1"]),
    ("CLM fine-tune (softce): TF-PCA state", ft_tf["acc"], ft_tf["f1"]),
    ("CLM(ft) + линейная голова (без TF)", head_clm_only["acc"], head_clm_only["f1"]),
    ("TF-голова (только TF-PCA)", head_tf_only["acc"], head_tf_only["f1"]),
    ("Франкенштейн: CLM(ft) + TimesFM → голова", head_frank["acc"], head_frank["f1"]),
]

print("=" * 74)
print(f"{'Подход':<52}{'acc':>8}{'F1_macro':>10}")
print("-" * 74)
for name, a, f in rows:
    if a is None or f is None:
        print(f"{name:<52}{'—':>8}{'—':>10}")
    else:
        print(f"{name:<52}{a:>8.4f}{f:>10.4f}")
print("=" * 74)

In [ ]:
# 6.2 Сводный бар-чарт по accuracy.
import numpy as np
import matplotlib.pyplot as plt

plot_items = [(r[0], r[1]) for r in rows if r[1] is not None]
labels = [p[0] for p in plot_items]
vals = [p[1] for p in plot_items]

plt.figure(figsize=(11, max(6, 0.55 * len(labels))))
bars = plt.barh(labels, vals, color="#4C72B0")
plt.xlabel("accuracy (test)")
plt.title("TimesFM → CLM: сравнение подходов")
plt.xlim(0, 1.0)
for b, v in zip(bars, vals):
    plt.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

### Что здесь вообще происходит — коротко

- **RF stats** — «честный» бейзлайн на тех же 11 статистиках, что идут в state.
- **Потолок на TimesFM** — RF/LogReg на сырых 1280-мерных эмбеддингах (после StandardScaler). Это максимум, чего признаки TimesFM способны достичь без CLM-механик.
- **CLM zero-shot** — «передали признаки в state модели», но головы CLM **не обучались** на этих классах; чистый перенос чекпойнта, обученного на Q&A и агентских траекториях (T-Rex, WikiRacing, ...). Скорее всего заметно хуже потолка.
- **CLM fine-tune (softce)** — головы обучены на train-сплите поверх замороженного Qwen3-8B (редкий, но лёгкий случай: backbone не трогаем, обучается ~20M параметров на закэшированных эмбеддингах). Ожидаемо сильно лучше zero-shot.
- **Франкенштейн** — то же самое, но признаки TimesFM идут **напрямую** в обучаемую голову рядом с 512-мерным вектором CLM: текст-бутылочное горлышко исключено.

### Важные оговорки

1. **Транспорт признаков через текст (tfmpca) — узкое горлышко:** 16 компонент PCA, округлённых до 3 знаков и прогнанных через BPE-токенизатор, несут лишь часть информации. Это честный «text bottleneck».
2. **Нет утечки:** PCA и StandardScaler калибруются только на train (ячейки 2.3–2.4).
3. **VRAM (T4 16GB):** Qwen3-8B грузится в 4-bit nf4 (~6 ГБ), эмбеддинги кэшируются на диск, после ячейки 4.3 энкодер выгружается. Если Colab не даёт 16 ГБ — поставьте в 3.2 пары `bnb_4bit_use_double_quant=True` и уменьшите `EMB_BS`.
4. **Верность рецепта:** эмбеддинги считаем как в тренировке CLM — без спец-токенов, last-token pool, L2-норм (то же, что даёт vLLM pooling-runner, но через transformers). Голова имеет смысл только со своим энкодером и пулингом — здесь это Qwen3-8B.
5. **Оптимизация:** для качества можно поднять число эпох/`lr`, добавить hard negatives, взять `--loss infonce` вместо `softce`, или включить raw-признаки в state помимо PCA (как для NanoJev).